# Day 19/42: Model Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week3_core_ml/day19_model_evaluation/day19_notebook.ipynb)

## What You'll Learn

- Why accuracy can be the most misleading metric you report
- Precision, Recall, and F1, and what each one actually protects against
- How changing the decision threshold trades precision for recall
- What ROC-AUC measures that a single accuracy number cannot
- How to pick the right metric for the cost of being wrong

## Datasets used

A synthetic imbalanced dataset built in this notebook with `make_classification`, no download needed. Imbalanced on purpose, because that's exactly where accuracy stops telling the truth.

Run every cell top to bottom. Nothing here needs an internet connection or an API key.

## Setup

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_curve, roc_auc_score,
                              precision_recall_curve, ConfusionMatrixDisplay)

np.random.seed(42)
print("Setup complete. Libraries loaded.")

## The Concept

Accuracy answers one question: out of all predictions, how many were correct? That sounds like the whole story. It isn't, the moment your classes are imbalanced.

**Precision**: out of everything the model flagged as positive, how much was actually positive? Low precision means false alarms.

**Recall**: out of everything that was actually positive, how much did the model catch? Low recall means missed cases.

**F1**: the harmonic mean of precision and recall. One number when you need both to matter, but it hides which one is failing.

**ROC-AUC**: how well the model ranks positives above negatives, across every possible threshold, not just the default 0.5 cutoff.

None of these is "the right metric." The right metric depends on what a false positive costs you versus what a false negative costs you. That's the whole lesson today.

## 1. The Accuracy Paradox

Build a dataset where 95% of cases belong to one class. Then let a model that learns absolutely nothing, `DummyClassifier`, just predict the majority class every single time.

In [ ]:
X, y = make_classification(
    n_samples=2000, n_features=10, n_informative=5,
    weights=[0.95, 0.05], flip_y=0.01, random_state=42
)

print("Class distribution:", pd.Series(y).value_counts().to_dict())
print("Majority class share:", round((y == 0).mean() * 100, 1), "%")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

print("Dummy classifier (always predicts the majority class, learns nothing):")
print("Accuracy: ", round(accuracy_score(y_test, dummy_pred), 3))
print("Recall:   ", round(recall_score(y_test, dummy_pred), 3))
print("Precision:", round(precision_score(y_test, dummy_pred, zero_division=0), 3))

A model with zero intelligence just scored roughly 95% accuracy. It also has 0% recall, it never once caught the minority class. If that minority class is fraud, cancer, or churn, a 95% accuracy headline is hiding a model that is functionally useless for the thing you actually built it for.

## 2. A Real Model Has the Same Blind Spot

Swap the dummy for an actual `LogisticRegression`. Default settings, default 0.5 threshold.

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print("Confusion matrix:\n", cm)
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\nAccuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")
print(f"F1:        {f1:.3f}")

In [ ]:
# Manual check, same formulas the metrics above use under the hood
manual_precision = tp / (tp + fp)
manual_recall = tp / (tp + fn)
manual_f1 = 2 * (manual_precision * manual_recall) / (manual_precision + manual_recall)

print(f"Manual precision: {manual_precision:.3f}")
print(f"Manual recall:    {manual_recall:.3f}")
print(f"Manual F1:        {manual_f1:.3f}")
print("\nMatches sklearn's numbers above. Precision and recall are simple division, nothing hidden.")

A real, trained logistic regression model lands at roughly 94-95% accuracy here too, almost identical to a model that learned nothing. But recall sits near 3%. The model is catching almost none of the minority class. Accuracy looked fine. The model is not fine.

## 3. Fixing Recall: class_weight='balanced'

`class_weight='balanced'` tells the model to penalize mistakes on the minority class harder during training, so it stops ignoring it.

In [ ]:
model_balanced = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
model_balanced.fit(X_train, y_train)
y_pred_bal = model_balanced.predict(X_test)

print("Balanced model:")
print("Accuracy: ", round(accuracy_score(y_test, y_pred_bal), 3))
print("Precision:", round(precision_score(y_test, y_pred_bal), 3))
print("Recall:   ", round(recall_score(y_test, y_pred_bal), 3))
print("F1:       ", round(f1_score(y_test, y_pred_bal), 3))

print("\nClassification report:")
print(classification_report(y_test, y_pred_bal, target_names=["majority(0)", "minority(1)"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0], cmap="Reds", colorbar=False)
axes[0].set_title("Default model (accuracy-optimized)")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_bal, ax=axes[1], cmap="Greens", colorbar=False)
axes[1].set_title("class_weight='balanced' (recall-optimized)")
plt.tight_layout()
plt.show()

Accuracy drops from roughly 94% to roughly 72%. Recall jumps from roughly 3% to roughly 75%. That's not the model getting worse. That's the model finally doing the job, at the cost of more false alarms (higher false positives). Whether that trade is worth it depends entirely on what a missed case costs you in the real world.

## 4. ROC-AUC: Evaluating Every Threshold at Once

Accuracy, precision, recall, and F1 all depend on one chosen threshold (usually 0.5). ROC-AUC asks a different question: across every possible threshold, how well does the model rank actual positives above actual negatives?

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6.5, 5))
plt.plot(fpr, tpr, color="#7C4DFF", linewidth=2, label=f"Model (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random guess (AUC = 0.5)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()

print(f"ROC-AUC: {auc:.3f}")

This is the same default model that had 3% recall at the 0.5 threshold. Its ROC-AUC is still meaningfully above 0.5, which means the model is actually ranking positives higher than negatives reasonably well. It just needs a lower threshold to act on that ranking. AUC tells you whether the model knows something useful. The threshold decides whether you use that knowledge.

## 5. Moving the Threshold: The Precision-Recall Tradeoff

Instead of accepting the default 0.5 cutoff, try several thresholds and watch precision and recall move in opposite directions.

In [ ]:
thresholds_to_try = [0.1, 0.2, 0.3, 0.5, 0.7]
threshold_results = []

for t in thresholds_to_try:
    preds_at_t = (y_proba >= t).astype(int)
    threshold_results.append({
        "threshold": t,
        "precision": round(precision_score(y_test, preds_at_t, zero_division=0), 3),
        "recall": round(recall_score(y_test, preds_at_t, zero_division=0), 3),
        "f1": round(f1_score(y_test, preds_at_t, zero_division=0), 3),
    })

pd.DataFrame(threshold_results)

In [ ]:
prec_curve, rec_curve, pr_thresholds = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(6.5, 5))
plt.plot(rec_curve, prec_curve, color="#00C4CC", linewidth=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.tight_layout()
plt.show()

Lower the threshold and the model flags more cases as positive: recall goes up, precision goes down, more false alarms slip in. Raise the threshold and the opposite happens. There is no threshold that maximizes both at once. Picking one is a business decision, not a modelling one: what does a false negative cost you, versus what does a false positive cost you?

## 6. Practice: Pick the Metric That Matters

Three systems below, each with a confusion matrix from 10,000 predictions. Before running the next cell, decide for each one: would you optimize for precision or recall, and why?

- **Spam filter**: flags an email as spam (positive class).
- **Cancer screening**: flags a scan as suspicious (positive class).
- **Fraud detector**: flags a transaction as fraudulent (positive class).

In [ ]:
practice_cases = [
    {"name": "Spam filter",    "tp": 180, "fp": 5,   "fn": 40, "tn": 9775},
    {"name": "Cancer screen",  "tp": 95,  "fp": 150, "fn": 5,  "tn": 9750},
    {"name": "Fraud detector", "tp": 60,  "fp": 300, "fn": 10, "tn": 9630},
]

rows = []
for case in practice_cases:
    tp, fp, fn, tn = case["tp"], case["fp"], case["fn"], case["tn"]
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    rows.append({
        "system": case["name"],
        "accuracy": round(accuracy, 3),
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3),
    })

pd.DataFrame(rows)

**Read it like this.**

The spam filter has high precision (0.97) and lower recall (0.82). That's correct for spam: a false positive means a real email lands in the spam folder and someone misses something important. Letting a few spam emails through (lower recall) is the safer failure.

Cancer screening has low precision (0.39) and very high recall (0.95) by design. A false positive means an extra follow-up test. A false negative means a missed cancer. The cost is wildly asymmetric, so the metric choice has to be too.

The fraud detector sits in between: precision is low (0.17) because flagging extra transactions for review is cheap, but recall is high (0.86) because missed fraud is expensive. All three systems would report "99%+ accuracy" if you only looked at that one number. None of that 99% tells you whether the system is doing its job.

## 7. Try It Yourself

No solution provided here. Make these changes and see what happens:

1. In Section 1, change `weights=[0.95, 0.05]` to `weights=[0.99, 0.01]`. Does the dummy classifier's accuracy go up or down? What happens to its recall?
2. In Section 5, add `threshold = 0.4` to the list and check where precision and recall cross over each other.
3. Pick your own cost ratio: imagine a false negative costs you 10x what a false positive costs. Which threshold from the table in Section 5 would you actually ship, and why?

## Self-Check Before Day 20

You're ready to move on if you can answer these without scrolling back up:

1. A fraud model reports 99.2% accuracy on a dataset that's 99% legitimate transactions. Is that impressive on its own?
2. What does lowering the classification threshold do to precision? To recall?
3. Why can two models have very different accuracy but the same ROC-AUC?
4. When would you choose a metric that optimizes for recall over precision? Give one example outside of this notebook.
5. What does F1 hide that precision and recall, viewed separately, would show you?

If any of these feel shaky, re-run the relevant section above before starting Day 20.

## What's Next

Tomorrow, Day 20: Cross Validation and Hyperparameter Tuning. A single train/test split, including everything you ran today, can get lucky or unlucky depending on which rows landed in which set. We'll fix that with K-Fold, then tune a model properly with GridSearchCV.

Full series repo: https://github.com/VaishnaviJagtap18/-42-Days-of-ML-Challenge

#42DaysOfML #MachineLearning #MLEngineer #Python #DataScience